#House Price Prediction Model

##Importing Data

In [98]:
import pandas as pd
import pyodbc

db_file_path = r'C:\Projekt\HousePriceML\HousePriceML.accdb'
conn = pyodbc.connect(r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=' + db_file_path)

cursor = conn.cursor()

cursor.execute('SELECT * FROM Housing')
columns = [column[0] for column in cursor.description]
data = cursor.fetchall()
rows = cursor.fetchall()

for row in rows:
    print(row)

df = pd.DataFrame.from_records(data, columns=columns)
cursor.close()
conn.close()


In [99]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,"""7229300521""","""20141013T000000""",231300.0,2,1.00,1180,5650,"""1""",0,0,...,7,1180,0,1955,0,"""98178""",47.5112,-122.257,1340,5650
1,"""6414100192""","""20141209T000000""",538000.0,3,2.25,2570,7242,"""2""",0,0,...,7,2170,400,1951,1991,"""98125""",47.7210,-122.319,1690,7639
2,"""5631500400""","""20150225T000000""",180000.0,2,1.00,770,10000,"""1""",0,0,...,6,770,0,1933,0,"""98028""",47.7379,-122.233,2720,8062
3,"""2487200875""","""20141209T000000""",604000.0,4,3.00,1960,5000,"""1""",0,0,...,7,1050,910,1965,0,"""98136""",47.5208,-122.393,1360,5000
4,"""1954400510""","""20150218T000000""",510000.0,3,2.00,1680,8080,"""1""",0,0,...,8,1680,0,1987,0,"""98074""",47.6168,-122.045,1800,7503


##Data Clean-up

In [100]:
df = df.drop('id', axis=1)

In [101]:
df['date'] = df['date'].str.strip('"')
df['floors'] = df['floors'].str.strip('"')
df['zipcode'] = df['zipcode'].str.strip('"')

In [102]:
df['floors'] = df['floors'].astype(float)
df['zipcode'] = df['zipcode'].astype(int)

In [103]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

In [104]:
df['date'] = df['date'].dt.year

In [105]:
df[['bedrooms', 'bathrooms']] = round(df[['bedrooms', 'bathrooms']], 0)

##Feature Engineering

In [106]:
df['age'] = df['date'] - df['yr_built']
df['bed_per_bath'] = df['bedrooms'] / (df['bathrooms'] + 1)
df['time_since_renovated'] = df['date'] - df['yr_renovated']
df['rooms'] = df['bathrooms'] + df['bedrooms']
df['sqft_per_room'] = df['sqft_living'] / (df['rooms'] + 1)

In [107]:
df = df.iloc[:, [0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,1]]

In [108]:
df.head()

,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,lat,long,sqft_living15,sqft_lot15,age,bed_per_bath,time_since_renovated,rooms,sqft_per_room,price
0,2014,2,1.0,1180,5650,1.0,0,0,3,7,...,47.5112,-122.257,1340,5650,59,1.0,2014,3.0,295.000000,231300.0
1,2014,3,2.0,2570,7242,2.0,0,0,3,7,...,47.7210,-122.319,1690,7639,63,1.0,23,5.0,428.333333,538000.0
2,2015,2,1.0,770,10000,1.0,0,0,3,6,...,47.7379,-122.233,2720,8062,82,1.0,2015,3.0,192.500000,180000.0
3,2014,4,3.0,1960,5000,1.0,0,0,5,7,...,47.5208,-122.393,1360,5000,49,1.0,2014,7.0,245.000000,604000.0
4,2015,3,2.0,1680,8080,1.0,0,0,3,8,...,47.6168,-122.045,1800,7503,28,1.0,2015,5.0,280.000000,510000.0


In [109]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  21613 non-null  int32  
 1   bedrooms              21613 non-null  int64  
 2   bathrooms             21613 non-null  float64
 3   sqft_living           21613 non-null  int64  
 4   sqft_lot              21613 non-null  int64  
 5   floors                21613 non-null  float64
 6   waterfront            21613 non-null  int64  
 7   view                  21613 non-null  int64  
 8   condition             21613 non-null  int64  
 9   grade                 21613 non-null  int64  
 10  sqft_above            21613 non-null  int64  
 11  sqft_basement         21613 non-null  int64  
 12  yr_built              21613 non-null  int64  
 13  yr_renovated          21613 non-null  int64  
 14  zipcode               21613 non-null  int64  
 15  lat                   21613 no

##Dataset Split

In [110]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
Y = df['price']

In [111]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [112]:
x_train.info()

<class 'pandas.DataFrame'>
Index: 17290 entries, 11689 to 7592
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  17290 non-null  int32  
 1   bedrooms              17290 non-null  int64  
 2   bathrooms             17290 non-null  float64
 3   sqft_living           17290 non-null  int64  
 4   sqft_lot              17290 non-null  int64  
 5   floors                17290 non-null  float64
 6   waterfront            17290 non-null  int64  
 7   view                  17290 non-null  int64  
 8   condition             17290 non-null  int64  
 9   grade                 17290 non-null  int64  
 10  sqft_above            17290 non-null  int64  
 11  sqft_basement         17290 non-null  int64  
 12  yr_built              17290 non-null  int64  
 13  yr_renovated          17290 non-null  int64  
 14  zipcode               17290 non-null  int64  
 15  lat                   17290 non-

##XGBoost

In [113]:
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators = 1500, learning_rate = 0.06, max_depth = 5)

#Training
model.fit (x_train, y_train, eval_set=[(x_test, y_test)], verbose=100)

#Prediction
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {mse}")

[0]	validation_0-rmse:357535.53203
[100]	validation_0-rmse:132746.98971
[200]	validation_0-rmse:128274.40238
[300]	validation_0-rmse:126124.45368
[400]	validation_0-rmse:125281.76854
[500]	validation_0-rmse:124605.78304
[600]	validation_0-rmse:124010.31690
[700]	validation_0-rmse:123489.69111
[800]	validation_0-rmse:123212.34327
[900]	validation_0-rmse:123156.46080
[1000]	validation_0-rmse:123114.49493
[1100]	validation_0-rmse:122867.49781
[1200]	validation_0-rmse:122788.08295
[1300]	validation_0-rmse:122723.01390
[1400]	validation_0-rmse:122663.87538
[1499]	validation_0-rmse:122581.27529
Mean Squared Error: 15026168936.450174


##Results


In [114]:
import numpy as np

results = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred,
    'Absolute Error': np.abs(y_test - y_pred),
    'Percentage Error': np.abs((y_test - y_pred) / y_test) * 100
})

formatted_results = results.copy()

formatted_results['Actual Price'] = formatted_results['Actual Price'].map('${:,.2f}'.format)
formatted_results['Predicted Price'] = formatted_results['Predicted Price'].map('${:,.2f}'.format)
formatted_results['Absolute Error'] = formatted_results['Absolute Error'].map('${:,.2f}'.format)
formatted_results['Percentage Error'] = formatted_results['Percentage Error'].map('{:,.2f}%'.format)

In [115]:
formatted_results.head(20)

,Actual Price,Predicted Price,Absolute Error,Percentage Error
999,"$289,000.00","$290,952.12","$1,952.12",0.68%
18168,"$252,000.00","$243,692.31","$8,307.69",3.30%
20553,"$344,000.00","$346,148.78","$2,148.78",0.62%
19082,"$550,000.00","$540,199.62","$9,800.38",1.78%
8311,"$500,000.00","$497,238.94","$2,761.06",0.55%
14616,"$333,000.00","$334,764.44","$1,764.44",0.53%
9293,"$327,500.00","$368,713.28","$41,213.28",12.58%
3256,"$1,208,000.00","$1,694,339.12","$486,339.12",40.26%
21293,"$208,400.00","$286,255.59","$77,855.59",37.36%
13517,"$845,000.00","$724,145.06","$120,854.94",14.30%


In [116]:
total_percentage_error = np.mean(results['Percentage Error'])
median_ape = np.median(results['Percentage Error'])

print(f"Median Percentage Error: {median_ape:.2f}%")
print(f'Mean Percentage Error: {total_percentage_error:.2f}%')

Median Percentage Error: 7.97%
Mean Percentage Error: 12.10%


In [117]:
formatted_results['Actual Price'].str.strip("$")
formatted_results['Predicted Price'].str.strip("$")

999      290,952.12
18168    243,692.31
20553    346,148.78
19082    540,199.62
8311     497,238.94
            ...    
9766     396,047.19
18177    572,251.88
1884     236,747.75
11967    325,735.19
18245    422,672.88
Name: Predicted Price, Length: 4323, dtype: str

In [119]:
formatted_results

,Actual Price,Predicted Price,Absolute Error,Percentage Error
999,"$289,000.00","$290,952.12","$1,952.12",0.68%
18168,"$252,000.00","$243,692.31","$8,307.69",3.30%
20553,"$344,000.00","$346,148.78","$2,148.78",0.62%
19082,"$550,000.00","$540,199.62","$9,800.38",1.78%
8311,"$500,000.00","$497,238.94","$2,761.06",0.55%
...,...,...,...,...
9766,"$499,000.00","$396,047.19","$102,952.81",20.63%
18177,"$590,000.00","$572,251.88","$17,748.12",3.01%
1884,"$150,000.00","$236,747.75","$86,747.75",57.83%
11967,"$325,000.00","$325,735.19",$735.19,0.23%


In [ ]:
formatted_results.to_csv("formatted_results.csv")